In [108]:
import numpy as np
from scipy.special import kn
from scipy.optimize import fsolve, root


In [109]:
!ls

T-n-1n0.dat          Untitled.ipynb       pdg2020
Untitled-Copy1.ipynb eos-3mu.ipynb


In [110]:
pdg2020=np.loadtxt("pdg2020",usecols=[1,2,3,4,5,6,7,8])
pdg2020[:,0]=pdg2020[:,0]/1000

In [111]:
def phi(m,g,mu,T):
    return g*m**2/(2*np.pi**2)*T*np.exp(mu/T)*kn(2,m/T)

In [121]:
pdg2020[0]

array([ 0.13957,  1.     ,  0.     ,  0.     ,  0.     , -1.     ,
        0.     ,  0.     ])

In [122]:
def ns(x,T):
    muB=x[0]
    muQ=x[1]
    muS=x[2]
    tnB1=0
    tnQ=0
    tnS=0
    tn=0
    for i,particle in enumerate(pdg2020):
        tmuB=particle[2]
        tmuQ=particle[3]
        tmuS=particle[4]
        tm=particle[0]
        tg=particle[1]
        tmu=(muB*tmuB+muQ*tmuQ+muS*tmuS)*.197
        tphi=phi(tm,tg,tmu,T*.197)
        tnB1+=tmuB*tphi
        tnQ+=tmuQ*tphi
        tnS+=tmuS*tphi
        tn+=tphi
        if (tmuB!=0 or tmuQ!=0 or tmuS!=0):
            tphi=phi(tm,tg,-tmu,T*.197)
            tnB1+=-tmuB*tphi
            tnQ+=-tmuQ*tphi
            tnS+=-tmuS*tphi
            tn+=tphi
    tnB1=tnB1/((0.197)**3)
    tnQ=tnQ/((0.197)**3)
    tnS=tnS/((0.197)**3)
    tn=tn/((0.197)**3)
    return np.array([tnB1,tnQ,tnS])

def nQs(muB,x,T):
    muQ=x[0]
    muS=x[1]
    tnB1=0
    tnQ=0
    tnS=0
    tn=0
    for i,particle in enumerate(pdg2020):
        tmuB=particle[2]
        tmuQ=particle[3]
        tmuS=particle[4]
        tm=particle[0]
        tg=particle[1]
        tmu=(muB*tmuB+muQ*tmuQ+muS*tmuS)*.197
        tphi=phi(tm,tg,tmu,T*.197)
        tnB1+=tmuB*tphi
        tnQ+=tmuQ*tphi
        tnS+=tmuS*tphi
        tn+=tphi
        if (tmuB!=0 or tmuQ!=0 or tmuS!=0):
            tphi=phi(tm,tg,-tmu,T*.197)
            tnB1+=-tmuB*tphi
            tnQ+=-tmuQ*tphi
            tnS+=-tmuS*tphi
            tn+=tphi
    tnB1=tnB1/((0.197)**3)
    tnQ=tnQ/((0.197)**3)
    tnS=tnS/((0.197)**3)
    tn=tn/((.197)**3)
    return tnB1,np.array([tnQ,tnS])

def nL(x,T):
    muB=x[0]
    muQ=x[1]
    muS=x[2]
    tnB1=0
    tnQ=0
    tnS=0
    tn=0
    for i,particle in enumerate(pdg2020):
        tmuB=particle[2]
        tmuQ=particle[3]
        tmuS=particle[4]
        tm=particle[0]
        tg=particle[1]
        tmu=(muB*tmuB+muQ*tmuQ+muS*tmuS)*.197
        tphi=phi(tm,tg,tmu,T*.197)
        tnB1+=tmuB*tphi
        tnQ+=tmuQ*tphi
        tnS+=tmuS*tphi
        tn+=tphi
        if (tmuB!=0 or tmuQ!=0 or tmuS!=0):
            tphi=phi(tm,tg,-tmu,T*.197)
            tnB1+=-tmuB*tphi
            tnQ+=-tmuQ*tphi
            tnS+=-tmuS*tphi
            tn+=tphi
    tnB1=tnB1/((0.197)**3)
    tnQ=tnQ/((0.197)**3)
    tnS=tnS/((0.197)**3)
    tn=tn/((.197)**3)
    return tn

def es(x,T):
    te=0
    
    muB=x[0]
    muQ=x[1]
    muS=x[2]
    for i,particle in enumerate(pdg2020):
        tmuB=particle[2]
        tmuQ=particle[3]
        tmuS=particle[4]
        tm=particle[0]
        tT=T*.197
        tg=particle[1]
        tmu=(muB*tmuB+muQ*tmuQ+muS*tmuS)*.197
        tphi=phi(tm,tg,tmu,T*.197)
        tpe=3*tT+tm*kn(1,tm/tT)/kn(2,tm/tT)
        te+=tpe*tphi
        if (tmuB!=0 or tmuQ!=0 or tmuS!=0):
            tphi=phi(tm,tg,-tmu,T*.197)
            te+=tpe*tphi
    return te/(.197**4)


In [123]:
def mus(rhos,T):
    def to_solve1(xk):
        d=ns(xk,T)
        return d-rhos
    return root(to_solve1,[.5,.5,.5],method="lm").x
    
def muQS(muB,T):
    def to_solve2(xk):
        nB1,d=nQs(muB,xk,T)
        return xk-np.array([nB1*0.4,0])
    return root(to_solve2,[.5,.5]).x

In [124]:
rhos=np.arange(3)
def to_solve1(xk):
    d=ns(xk,1)[:3]
    return d-rhos
to_solve1([1,2,3]).shape



(3,)

In [134]:
a=mus([0.16,.4*0.16,0.0],250/197)
print(a)
a.shape

[-5.58281759e-05  5.05293099e-05  2.20149277e-01]


(3,)

In [133]:
a=muQS(0.16,250/197)
print(a)

[0. 0.]


In [107]:
es([.16,a[0],a[1]],230/197)

1.3778485490890499e-17

In [21]:
#HRG EoS

def p(muB,T):
    muQ, muS = muQS(muB,T)
    tnL=nL([muB,muQ,muS],T)
    return T*tnL
def nB(muB,T):
    muQ, muS = muQS(muB,T)
    nB,nQ,nS=ns([muB,muQ,muS],T)
    return nB
def e(muB,T):
    muQ, muS = muQS(muB,T)
    return es([muB,muQ,muS],T)
def dT_p(muB,T):
    dT=1e-5
    return (p(muB,T+dT)-p(muB,T-dT))/(2*dT)
def dmu_e(muB,T):
    dmu=1e-5
    return (e(muB+dmu,T)-e(muB-dmu,T))/(2*dmu)
def dmu_n(muB,T):
    dmu=1e-5
    return (nB(muB+dmu,T)-nB(muB-dmu,T))/(2*dmu)
def dChi_uds(x,T):
    dmu=1e-5
    chis=np.zeros((3,3), dtype=np.float32)
    #for i in range(3):
    for j in range(3):
        tx1=x
        tx1[j]=tx1[j]+dmu
        tx2=x
        tx2[j]=tx2[j]-dmu
        tchi=(ns(tx1,T)-ns(tx2,T))/(2*dmu)
        chis[j]=tchi
    return chis
def dT_n(muB,T):
    dT=1e-5
    return (nB(muB,T+dT)-nB(muB,T-dT))/(2*dT)
def dT_e(muB,T):
    dT=1e-5
    return (e(muB,T+dT)-e(muB,T-dT))/(2*dT)
def cs2(muB,T):
    return dT_p(muB,T)/(dT_e(muB,T)-dmu_e(muB,T)*(dT_n(muB,T)/dmu_n(muB,T)))+nB(muB,T)/(dmu_e(muB,T)-dT_e(muB,T)*(dmu_n(muB,T)/dT_n(muB,T)))
def crho2(muB,T):
    return dT_p(muB,T)/(dT_n(muB,T)-dmu_n(muB,T)*(dT_e(muB,T)/dmu_e(muB,T)))+nB(muB,T)/(dmu_n(muB,T)-dT_n(muB,T)*(dmu_e(muB,T)/dT_e(muB,T)))

In [ ]:
def dChi_uds(x,T):
    dmu=1e-4
    chis=np.zeros((3,3), dtype=np.float32)
    #for i in range(3):
    for j in range(3):
        tx1=x.copy()
        tx1[j]=tx1[j]+dmu
        tx2=x.copy()
        tx2[j]=tx2[j]-dmu
        tchi=(ns(tx1,T)-ns(tx2,T))/(2*dmu)
        chis[j]=tchi
    return chis
dChi_uds(mus([.16,.16*.4,0],Ts[0]),Ts[0])

#Derivatives of densities

def sounds(muB,T):
    muQ, muS = muQS(muB,T)
    dx=1e-5
    drhodmudT=np.zeros((4,4),dtype=np.float64)
    x=np.array([muB,muQ,muS])
    drhodmudT[:3,:3]=dChi_uds(x,T)
    for i in range(3):
        tdx=np.zeros(3,dtype=np.float64)
        tdx[i]=dx
        tx1=x+tdx
        tx2=x-tdx
        drhodmudT[3,i]=(es(tx1,T)-es(tx2,T))/(2*dx)+
        
    return np.array([tnB1,tnQ,tnS])
 

In [624]:
#times
dt=0.003125
t0=1
tfin=11

#Initial density
rho0=1*.16
#initial temperature
T0=180/197
#derived

#Initial chemical potential

from scipy.optimize import fsolve
mu0=mus([rho0,.4*rho0,0],T0)
print(mu0)
def smuB(i0,T):
    return mus([i0,.4*i0,0],T)[0]
#def smuB(i0,T):
#    return i0/(2*pol_p2(T)*T**3)
def dT_smuB(i0,T):
    dT=1e-5
    return (smuB(i0,T+dT)-smuB(i0,T-dT))/(2*dT)
muB0=smuB(rho0,T0)
e0=e(muB0,T0)
p0=p(muB0,T0)
s0=e0+p0-muB0*rho0
cs20=cs2(muB0,T0)
crho20=crho2(muB0,T0)
print("e0: {}".format(e0))

print("T0: {}".format(T0))

print("mu0: {}".format(mu0))

print("rho0: {}".format(rho0))

print("p0: {}".format(p0))

print("s0: {}".format(s0))

print("cs2: {}".format(cs20))
print("crho2: {}".format(crho20))
def rho(tau):
    return rho0 * t0/tau
def dtau_rho(tau):
    return -rho0*t0/tau**2

[ 0.75121135 -0.03248801  0.23154462]
e0: 5.978601513534713
T0: 0.9137055837563451
mu0: [ 0.75121135 -0.03248801  0.23154462]
rho0: 0.16
p0: 0.8623867488613255
s0: 6.720794445652328
cs2: 0.14786755848399902
crho2: -0.29297381452695537


In [625]:
ns([0.25809904, -0.01122065,  0.07818563], T0)

array([ 5.09295816e-02,  2.03718315e-02, -1.42707037e-09])

In [626]:
def f_ode(xc,tau):
    y=mus(np.array([rho(tau),0.4*rho(tau),0]),xc)[0]
    A=-((e(y,xc)+p(y,xc))/tau+dmu_e(y,xc)/dmu_n(y,xc)*dtau_rho(tau))/(dT_e(y,xc)+dmu_e(y,xc)*dT_smuB(rho(tau),xc))
    return A

In [627]:
f_ode(230/197,1)
import matplotlib.pyplot as plt

array([[ 2.9701526,  0.7271029, -1.5208354],
       [ 0.7271029,  4.2401614,  1.0139514],
       [-1.5208354,  1.0139514,  3.552511 ]], dtype=float32)

In [632]:
ts=np.linspace(1.0,11,100)
dt=10/99
Ts=np.zeros(100,dtype=np.float32)
Ts[0]=250/197
for i in np.arange(1,100):
    print(f_ode(Ts[i-1],ts[i-1]))
    tTs=Ts[i-1]+dt*f_ode(Ts[i-1],ts[i])
    print(tTs)
    Ts[i]=Ts[i-1]+dt/2*(f_ode(Ts[i-1],ts[i-1])+f_ode(tTs,ts[i]))

-0.18640014100167515
1.2519228993677174
-0.16629745213636346
1.2358233827348668
-0.14984938089871322
1.2212775836395233
-0.13617303100715034
1.2080276582933893
-0.12464159239560552
1.1958736742506237
-0.11480019612729572
1.1846574742775793
-0.10631191401007839
1.1742524207375873
-0.0989223632978628
1.1645551051045293
-0.09243625559016783
1.1554805171119478
-0.08670133438299507
1.1469576472302057
-0.081597287634491
1.1389268735955702
-0.07702783814028807
1.1313373553409953
-0.07291510607646987
1.12414568525569
-0.06919544505217376
1.117314431759307
-0.06581635816967198
1.1108110745962214
-0.06273418278920202
1.1046072383333603
-0.059912320760612474
1.098678002354934
-0.05731987242364491
1.0930012980775712
-0.054930586941217226
1.0875576382122047
-0.052722017960323615
1.082329591383913
-0.050674869400460086
1.0773015986210246
-0.04877246888160383
1.0724597794510844
-0.04700032886166783
1.0677914925638416
-0.04534581177295482
1.063285364320817
-0.0437978434440365
1.0589310750204632
-0.042

In [645]:
sol=Ts
ts=np.linspace(1.0,11,100)

In [651]:
np.savetxt("T-n-1n0.dat",np.stack((ts,sol)))
s_mu=np.zeros(ts.size,dtype=np.float32)
for i,tT in enumerate(sol):
    s_mu[i]=smuB(rho(ts[i]),tT)
print(s_mu)


np.savetxt("p-n-1n0.dat",p(s_mu,sol))
np.savetxt("e-n-1n0.dat",e(s_mu,sol))
np.savetxt("s-n-1n0.dat",e(s_mu,sol)+s_mu*rho(ts)+p(s_mu,sol))
np.savetxt("cs2-n-1n0.dat",cs2(s_mu,sol))
np.savetxt("crho2-n-1n0.dat",crho2(s_mu,sol))
np.savetxt("mu-n-1n0.dat",s_mu)
np.savetxt("w-n-1n0.dat",dmu_n(s_mu,sol))

[0.07144745 0.07147874 0.07152755 0.07158911 0.07166035 0.07173923
 0.07182418 0.07191411 0.07200812 0.07210554 0.0722058  0.07230851
 0.07241328 0.07251982 0.07262787 0.07273722 0.07284769 0.07295913
 0.07307141 0.07318443 0.0732981  0.07341229 0.07352695 0.07364201
 0.07375742 0.07387318 0.07398917 0.07410535 0.07422176 0.07433835
 0.07445503 0.07457185 0.07468873 0.07480565 0.07492262 0.07503963
 0.07515668 0.07527374 0.07539081 0.07550785 0.07562485 0.07574181
 0.07585873 0.07597558 0.0760924  0.07620914 0.07632582 0.07644242
 0.07655898 0.07667547 0.07679185 0.07690814 0.07702436 0.0771405
 0.07725655 0.07737251 0.07748841 0.0776042  0.0777199  0.07783552
 0.07795102 0.07806643 0.07818174 0.07829696 0.07841209 0.07852714
 0.07864209 0.07875695 0.07887171 0.07898638 0.07910095 0.07921543
 0.07932979 0.07944408 0.07955827 0.07967235 0.07978637 0.07990029
 0.08001411 0.08012783 0.08024146 0.08035502 0.08046846 0.08058184
 0.08069512 0.08080831 0.08092143 0.08103445 0.08114739 0.08126

<ipython-input-618-717993e15bed>:10: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return xk-np.array([nB1*0.4,0])


TypeError: Cannot cast array data from dtype('O') to dtype('float64') according to the rule 'safe'

error: Result from function call is not a proper array of floats.

In [644]:
ts.shape

(10,)

In [649]:
np.stack((ts,p(s_mu,sol)))

<ipython-input-618-717993e15bed>:10: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return xk-np.array([nB1*0.4,0])


TypeError: Cannot cast array data from dtype('O') to dtype('float64') according to the rule 'safe'

error: Result from function call is not a proper array of floats.